In [1]:
%pip install pandas numpy requests clickhouse-connect sqlalchemy openpyxl


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from enum import unique

from numpy.ma.core import append
import requests
import pandas as pd

url = "https://BrsApi.ir/Api/Tsetmc/AllSymbols.php?key=BusnGbvtCSfYqNKW2B5TXFHm96mnT98A"

In [3]:
headers = {

        "User-Agent": "Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36 OPR/106.0.0.0",

        "Accept": "application/json, text/plain, */*"
        }

In [4]:
response = requests.get(url, headers=headers)

In [5]:
if response.status_code == 200:
    try:
        # Parse JSON data
        data = response.json()

        # If the data is a list of records, convert directly
        df = pd.DataFrame(data)
        print(df.head())  # Show first few rows
        # Now you can work with the DataFrame: save to CSV, analyze, etc.
    except ValueError:
        print("Response is not valid JSON")
        print(response.text)
else:
    print(f"Error: {response.status_code}")
    print(response.text)

       time      l18                     l30          isin                 id  \
0  12:09:22    وبملت                بانک ملت  IRO1BMLT0001    778253364357513   
1  12:09:24   وتجارت              بانک تجارت  IRO1BTEJ0001  63917421733088077   
2  12:09:24     شپنا       پالایش نفت اصفهان  IRO1PNES0001   7745894403636165   
3  12:09:23  وفیروزه  گروه توسعه مالی فیروزه  IRO1VFRZ0001   2160777579058191   
4  12:09:24    خساپا                   سایپا  IRO1SIPA0001  44891482026867833   

                                    cs  cs_id              z  bvol  \
0             بانک‌ها و موسسات اعتباری     57  3382000000000     1   
1             بانک‌ها و موسسات اعتباری     57  1590603130011     1   
2  فراورده‌های نفتی، کک و سوخت هسته‌ای     23   627234213000     1   
3                      سرمایه‌گذاری‌ها     56    18000000000     1   
4                   خودرو و ساخت قطعات     34  1621144826000     1   

                 mv  ...   pd4   po4      qo4  zo4  zd5     qd5   pd5   po5  \
0  4498060000

In [6]:
cols = [
    'time', 'l18', 'tvol', 'tno', 'tval', 'pc', 'pcp', 'plp',
    'Buy_CountI', 'Buy_CountN', 'Sell_CountI', 'Sell_CountN',
    'Buy_I_Volume', 'Buy_N_Volume', 'Sell_I_Volume', 'Sell_N_Volume', 'cs'
]
df = df[cols]
df

,time,l18,tvol,tno,tval,pc,pcp,plp,Buy_CountI,Buy_CountN,Sell_CountI,Sell_CountN,Buy_I_Volume,Buy_N_Volume,Sell_I_Volume,Sell_N_Volume,cs
0,12:09:22,وبملت,12932926332,89081,17198095305170,1330,-2.71,-3.00,16224,80,8567,84,7102952294,5829454038,9844585554,3087820778,بانک‌ها و موسسات اعتباری
1,12:09:24,وتجارت,14115597068,72372,9175191819312,650,1.09,-2.80,4707,76,4708,72,9893597114,4209536065,11212511130,2890622049,بانک‌ها و موسسات اعتباری
2,12:09:24,شپنا,3946060750,64772,35990517857660,9120,2.47,2.92,11697,300,11500,180,2677062367,1267582760,2943730553,1000914574,فراورده‌های نفتی، کک و سوخت هسته‌ای
3,12:09:23,وفیروزه,121023602,53315,412811506422,3411,2.99,2.99,324,15,52603,17,112889675,8000000,60473163,60416512,سرمایه‌گذاری‌ها
4,12:09:24,خساپا,7977825512,46741,4688640456785,588,1.03,-2.75,4539,24,3555,22,7150539819,820377104,6346347494,1624569429,خودرو و ساخت قطعات
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1118,06:10:16,وآیند,0,0,0,8950,0.00,0.00,0,0,0,0,0,0,0,0,بانک‌ها و موسسات اعتباری
1119,06:10:22,پویا,0,0,0,653121,0.00,0.00,0,0,0,0,0,0,0,0,صندوق سرمایه‌گذاری قابل معامله
1120,06:10:22,ونچر,0,0,0,1514000,0.00,0.00,0,0,0,0,0,0,0,0,صندوق سرمایه‌گذاری قابل معامله
1121,06:10:22,کمان,0,0,0,550000,0.00,0.00,0,0,0,0,0,0,0,0,صندوق سرمایه‌گذاری قابل معامله


In [8]:
df['time'] = pd.to_datetime(df['time'])

df['year_month'] = df['time'].dt.to_period('M')
df['monthly_avg_volume'] = df.groupby(['l18', 'year_month'])['tvol'].transform('mean')

df['Buy_AvgI'] = df['Buy_I_Volume'] / df['Buy_CountI'].replace(0, pd.NA)
df['Buy_AvgN'] = df['Buy_N_Volume'] / df['Buy_CountN'].replace(0, pd.NA)
df['Sell_AvgI'] = df['Sell_I_Volume'] / df['Sell_CountI'].replace(0, pd.NA)
df['Sell_AvgN'] = df['Sell_N_Volume'] / df['Sell_CountN'].replace(0, pd.NA)

total_buy = df['Buy_I_Volume'] + df['Buy_N_Volume']
total_sell = df['Sell_I_Volume'] + df['Sell_N_Volume']
df['Buy_PctN'] = (df['Buy_N_Volume'] / total_buy.replace(0, pd.NA)) * 100
df['Buy_PctI'] = (df['Buy_I_Volume'] / total_buy.replace(0, pd.NA)) * 100
df['Sell_PctN'] = (df['Sell_N_Volume'] / total_sell.replace(0, pd.NA)) * 100
df['Sell_PctI'] = (df['Sell_I_Volume'] / total_sell.replace(0, pd.NA)) * 100

# دو خط جدید برای نسبت خریدار به فروشنده
df['Buy_Sell_RatioI'] = df['Buy_I_Volume'] / df['Sell_I_Volume'].replace(0, pd.NA)
df['Buy_Sell_RatioN'] = df['Buy_N_Volume'] / df['Sell_N_Volume'].replace(0, pd.NA)

print(df.head())

                 time      l18         tvol    tno            tval    pc  \
0 2026-06-21 12:09:22    وبملت  12932926332  89081  17198095305170  1330   
1 2026-06-21 12:09:24   وتجارت  14115597068  72372   9175191819312   650   
2 2026-06-21 12:09:24     شپنا   3946060750  64772  35990517857660  9120   
3 2026-06-21 12:09:23  وفیروزه    121023602  53315    412811506422  3411   
4 2026-06-21 12:09:24    خساپا   7977825512  46741   4688640456785   588   

    pcp   plp  Buy_CountI  Buy_CountN  ...        Buy_AvgI         Buy_AvgN  \
0 -2.71 -3.00       16224          80  ...   437805.244946     72868175.475   
1  1.09 -2.80        4707          76  ...  2101890.187805  55388632.434211   
2  2.47  2.92       11697         300  ...   228867.433273   4225275.866667   
3  2.99  2.99         324          15  ...    348424.92284    533333.333333   
4  1.03 -2.75        4539          24  ...  1575355.765367  34182379.333333   

        Sell_AvgI        Sell_AvgN   Buy_PctN   Buy_PctI  Sell_PctN 

In [9]:
drop_cols = [
    'Buy_CountI', 'Buy_CountN', 'Sell_CountI', 'Sell_CountN',
    'Buy_I_Volume', 'Buy_N_Volume', 'Sell_I_Volume', 'Sell_N_Volume',
    'year_month'
]
df = df.drop(columns=drop_cols)
print(df.head())

                 time      l18         tvol    tno            tval    pc  \
0 2026-06-21 12:09:22    وبملت  12932926332  89081  17198095305170  1330   
1 2026-06-21 12:09:24   وتجارت  14115597068  72372   9175191819312   650   
2 2026-06-21 12:09:24     شپنا   3946060750  64772  35990517857660  9120   
3 2026-06-21 12:09:23  وفیروزه    121023602  53315    412811506422  3411   
4 2026-06-21 12:09:24    خساپا   7977825512  46741   4688640456785   588   

    pcp   plp                                   cs  monthly_avg_volume  \
0 -2.71 -3.00             بانک‌ها و موسسات اعتباری        1.293293e+10   
1  1.09 -2.80             بانک‌ها و موسسات اعتباری        1.411560e+10   
2  2.47  2.92  فراورده‌های نفتی، کک و سوخت هسته‌ای        3.946061e+09   
3  2.99  2.99                      سرمایه‌گذاری‌ها        1.210236e+08   
4  1.03 -2.75                   خودرو و ساخت قطعات        7.977826e+09   

         Buy_AvgI         Buy_AvgN       Sell_AvgI        Sell_AvgN  \
0   437805.244946     72868

In [10]:
rename_map = {
    'time': 'تاریخ (time)',
    'l18': 'نماد (l18)',
    'monthly_avg_volume': 'میانگین حجم معاملات (monthly_avg_volume)',
    'tvol': 'حجم معاملات (tvol)',
    'tval': 'ارزش معاملات (tval)',
    'pc': 'قیمت پایانی (pc)',
    'Buy_AvgI': 'سرانه خرید حقیقی (Buy_AvgI)',
    'Buy_AvgN': 'سرانه خرید حقوقی (Buy_AvgN)',
    'Sell_AvgI': 'سرانه فروش حقیقی (Sell_AvgI)',
    'Sell_AvgN': 'سرانه فروش حقوقی (Sell_AvgN)',
    'Buy_Sell_RatioI': 'نسبت حقیقی (Buy_Sell_RatioI)',
    'Buy_Sell_RatioN': 'نسبت حقوقی (Buy_Sell_RatioN)',
    'Buy_PctI': 'درصد خرید حقیقی (Buy_PctI)',
    'Buy_PctN': 'درصد خرید حقوقی (Buy_PctN)',
    'Sell_PctI': 'درصد فروش حقیقی (Sell_PctI)',
    'Sell_PctN': 'درصد فروش حقوقی (Sell_PctN)',
    'pcp': 'درصد پایانی (pcp)',
    'plp': 'درصد آخرین قیمت (plp)',
    'cs' : 'گروه'
}

new_order = [
    'تاریخ (time)',
    'نماد (l18)',
    'حجم معاملات (tvol)',
    'میانگین حجم معاملات (monthly_avg_volume)',
    'ارزش معاملات (tval)',
    'قیمت پایانی (pc)',
    'سرانه خرید حقیقی (Buy_AvgI)',
    'سرانه خرید حقوقی (Buy_AvgN)',
    'سرانه فروش حقیقی (Sell_AvgI)',
    'سرانه فروش حقوقی (Sell_AvgN)',
    'نسبت حقیقی (Buy_Sell_RatioI)',
    'نسبت حقوقی (Buy_Sell_RatioN)',
    'درصد خرید حقیقی (Buy_PctI)',
    'درصد خرید حقوقی (Buy_PctN)',
    'درصد فروش حقیقی (Sell_PctI)',
    'درصد فروش حقوقی (Sell_PctN)',
    'درصد پایانی (pcp)',
    'درصد آخرین قیمت (plp)',
    'گروه'
]

df = df.rename(columns=rename_map)
df = df[new_order]
print(df.head())

         تاریخ (time) نماد (l18)  حجم معاملات (tvol)  \
0 2026-06-21 12:09:22      وبملت         12932926332   
1 2026-06-21 12:09:24     وتجارت         14115597068   
2 2026-06-21 12:09:24       شپنا          3946060750   
3 2026-06-21 12:09:23    وفیروزه           121023602   
4 2026-06-21 12:09:24      خساپا          7977825512   

   میانگین حجم معاملات (monthly_avg_volume)  ارزش معاملات (tval)  \
0                              1.293293e+10       17198095305170   
1                              1.411560e+10        9175191819312   
2                              3.946061e+09       35990517857660   
3                              1.210236e+08         412811506422   
4                              7.977826e+09        4688640456785   

   قیمت پایانی (pc) سرانه خرید حقیقی (Buy_AvgI) سرانه خرید حقوقی (Buy_AvgN)  \
0              1330               437805.244946                72868175.475   
1               650              2101890.187805             55388632.434211   
2              91

In [11]:
df

,تاریخ (time),نماد (l18),حجم معاملات (tvol),میانگین حجم معاملات (monthly_avg_volume),ارزش معاملات (tval),قیمت پایانی (pc),سرانه خرید حقیقی (Buy_AvgI),سرانه خرید حقوقی (Buy_AvgN),سرانه فروش حقیقی (Sell_AvgI),سرانه فروش حقوقی (Sell_AvgN),نسبت حقیقی (Buy_Sell_RatioI),نسبت حقوقی (Buy_Sell_RatioN),درصد خرید حقیقی (Buy_PctI),درصد خرید حقوقی (Buy_PctN),درصد فروش حقیقی (Sell_PctI),درصد فروش حقوقی (Sell_PctN),درصد پایانی (pcp),درصد آخرین قیمت (plp),گروه
0,2026-06-21 12:09:22,وبملت,12932926332,1.293293e+10,17198095305170,1330,437805.244946,72868175.475,1149128.697794,36759771.166667,0.721509,1.887886,54.923671,45.076329,76.123386,23.876614,-2.71,-3.00,بانک‌ها و موسسات اعتباری
1,2026-06-21 12:09:24,وتجارت,14115597068,1.411560e+10,9175191819312,650,2101890.187805,55388632.434211,2381586.90102,40147528.458333,0.882371,1.456273,70.151767,29.848233,79.503689,20.496311,1.09,-2.80,بانک‌ها و موسسات اعتباری
2,2026-06-21 12:09:24,شپنا,3946060750,3.946061e+09,35990517857660,9120,228867.433273,4225275.866667,255976.569826,5560636.522222,0.909411,1.266425,67.865734,32.134266,74.625992,25.374008,2.47,2.92,فراورده‌های نفتی، کک و سوخت هسته‌ای
3,2026-06-21 12:09:23,وفیروزه,121023602,1.210236e+08,412811506422,3411,348424.92284,533333.333333,1149.614338,3553912.470588,1.866773,0.132414,93.382396,6.617604,50.023431,49.976569,2.99,2.99,سرمایه‌گذاری‌ها
4,2026-06-21 12:09:24,خساپا,7977825512,7.977826e+09,4688640456785,588,1575355.765367,34182379.333333,1785189.168495,73844064.954545,1.126717,0.504981,89.70787,10.29213,79.618789,20.381211,1.03,-2.75,خودرو و ساخت قطعات
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1118,2026-06-21 06:10:16,وآیند,0,0.000000e+00,0,8950,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.00,0.00,بانک‌ها و موسسات اعتباری
1119,2026-06-21 06:10:22,پویا,0,0.000000e+00,0,653121,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.00,0.00,صندوق سرمایه‌گذاری قابل معامله
1120,2026-06-21 06:10:22,ونچر,0,0.000000e+00,0,1514000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.00,0.00,صندوق سرمایه‌گذاری قابل معامله
1121,2026-06-21 06:10:22,کمان,0,0.000000e+00,0,550000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.00,0.00,صندوق سرمایه‌گذاری قابل معامله


In [13]:
print(df["گروه"].nunique())

49


In [9]:
cs = list(df["cs"].unique())
cs

['صندوق سرمایه\u200cگذاری قابل معامله',
 'محصولات شیمیایی',
 'ماشین\u200cآلات و تجهیزات',
 'فلزات اساسی',
 'اطلاعات و ارتباطات',
 'بیمه و صندوق بازنشستگی به جز تامین اجتماعی',
 'سیمان، آهک و گچ',
 'استخراج کانه\u200cهای فلزی',
 'بانک\u200cها و موسسات اعتباری',
 'حمل و نقل، انبارداری و ارتباطات',
 'انبوه\u200cسازی، املاک و مستغلات',
 'سرمایه\u200cگذاری\u200cها',
 'محصولات غذایی و آشامیدنی به جز قند و شکر',
 'شرکت\u200cهای چند رشته\u200cای صنعتی',
 'خودرو و ساخت قطعات',
 'قند و شکر',
 'ساخت محصولات فلزی',
 'رایانه و فعالیت\u200cهای وابسته به آن',
 'مواد و محصولات دارویی',
 'زراعت و خدمات وابسته',
 'سایر محصولات کانی غیرفلزی',
 'ساخت دستگاه\u200cها و وسایل ارتباطی',
 'سایر واسطه\u200cگری\u200cهای مالی',
 'لاستیک و پلاستیک',
 'فراورده\u200cهای نفتی، کک و سوخت هسته\u200cای',
 'پیمانکاری صنعتی',
 'استخراج زغال سنگ',
 'کاشی و سرامیک',
 'هتل و رستوران',
 'فعالیت\u200cهای کمکی به نهادهای مالی واسط',
 'تولید محصولات کامپیوتری الکترونیکی و نوری',
 'محصولات کاغذی',
 'عرضه برق، گاز، بخار و آب گرم',

In [14]:
df.columns

Index(['تاریخ (time)', 'نماد (l18)', 'حجم معاملات (tvol)',
       'میانگین حجم معاملات (monthly_avg_volume)', 'ارزش معاملات (tval)',
       'قیمت پایانی (pc)', 'سرانه خرید حقیقی (Buy_AvgI)',
       'سرانه خرید حقوقی (Buy_AvgN)', 'سرانه فروش حقیقی (Sell_AvgI)',
       'سرانه فروش حقوقی (Sell_AvgN)', 'نسبت حقیقی (Buy_Sell_RatioI)',
       'نسبت حقوقی (Buy_Sell_RatioN)', 'درصد خرید حقیقی (Buy_PctI)',
       'درصد خرید حقوقی (Buy_PctN)', 'درصد فروش حقیقی (Sell_PctI)',
       'درصد فروش حقوقی (Sell_PctN)', 'درصد پایانی (pcp)',
       'درصد آخرین قیمت (plp)', 'گروه'],
      dtype='str')